## imports

In [1]:
import os
import warnings
import logging
from typing import Tuple, Dict
from pypdf import PdfReader
from transformers import pipeline, AutoTokenizer, BitsAndBytesConfig
from accelerate import infer_auto_device_map
from transformers import AutoModelForCausalLM
import torch
import gc

# Suppress pypdf warnings about wrong pointing objects
logging.getLogger('pypdf').setLevel(logging.ERROR)

# model setup 

In [2]:
# Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-1b-it",
    quantization_config=quantization_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

# Verify quantization was applied
print(f"\nModel loaded successfully!")
if torch.cuda.is_available():
    print(f"GPU Memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"GPU Memory reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    #max_new_tokens=1024,
    batch_size=1,  # Process one at a time to reduce memory
    return_full_text=False,
    do_sample=True,  # Enable sampling instead of greedy decoding
    temperature=0.7,  # Control randomness (0.7 is good balance)
    top_p=0.9,  # Nucleus sampling
    repetition_penalty=1.2  # Prevent repetitive output
)


Loading model with 4-bit quantization...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

/home/viper/Documents/GitHub/COMP5700-Project/.venv/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Model loaded successfully!
GPU Memory allocated: 0.90 GB
GPU Memory reserved: 0.94 GB


Passing `generation_config` together with generation-related arguments=({'top_p', 'temperature', 'repetition_penalty', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


# I/O functions

## read two documents function

In [3]:
def input_documents(pdf1_path: str, pdf2_path: str) -> Tuple[str, str]:
    """
    Load and validate two PDF documents.
    
    Args:
        pdf1_path: Path to the first PDF document
        pdf2_path: Path to the second PDF document
    
    Returns:
        Tuple containing the text content from both PDFs
    
    Raises:
        FileNotFoundError: If either PDF file does not exist
        ValueError: If either file is not a valid PDF or cannot be read
    """
    # Validate that both files exist
    if not os.path.exists(pdf1_path):
        raise FileNotFoundError(f"First PDF not found: {pdf1_path}")
    if not os.path.exists(pdf2_path):
        raise FileNotFoundError(f"Second PDF not found: {pdf2_path}")
    
    # Validate file extensions
    if not pdf1_path.lower().endswith('.pdf'):
        raise ValueError(f"First file is not a PDF: {pdf1_path}")
    if not pdf2_path.lower().endswith('.pdf'):
        raise ValueError(f"Second file is not a PDF: {pdf2_path}")
    
    # Extract text from first PDF
    try:
        reader1 = PdfReader(pdf1_path)
        text1 = ""
        for page in reader1.pages:
            text1 += page.extract_text()
        
        if not text1.strip():
            raise ValueError(f"First PDF contains no extractable text: {pdf1_path}")
    except Exception as e:
        raise ValueError(f"Failed to read first PDF {pdf1_path}: {str(e)}")
    
    # Extract text from second PDF
    try:
        reader2 = PdfReader(pdf2_path)
        text2 = ""
        for page in reader2.pages:
            text2 += page.extract_text()
        
        if not text2.strip():
            raise ValueError(f"Second PDF contains no extractable text: {pdf2_path}")
    except Exception as e:
        raise ValueError(f"Failed to read second PDF {pdf2_path}: {str(e)}")
    
    return text1, text2

## print LLM output function

In [4]:
def PrintPrompt(prompt, promptType: str, LLMoutput):
    with open('LLMoutputs.txt', 'a', encoding='utf-8') as f:
        f.write(model.config._name_or_path) #model name
        f.write(f"\n...\n")
        f.write(prompt) # prompt used
        f.write(f"\n...\n")
        f.write(promptType) # prompt type
        f.write(f"\n...\n")
        f.write(str(LLMoutput)) # LLM output
        f.write(f"\n...\n")
        f.flush()  # Ensure data is written immediately
        print(f"print successfull")

# prompt functions

## zero shot

In [ ]:
def zero_shot_prompt(doc1: str, doc2: str, max_chars: int = 60000) -> str:
    """
    Construct a zero-shot prompt to identify key data elements in two input documents.
    
    Args:
        doc1: Text content of the first document
        doc2: Text content of the second document
        max_chars: Maximum characters per document to avoid memory issues (default: 2000)
    
    Returns:
        A string containing the zero-shot prompt for the LLM
    """
    # Truncate documents significantly to avoid OOM errors
    doc1_truncated = doc1[:max_chars] + ("..." if len(doc1) > max_chars else "")
    doc2_truncated = doc2[:max_chars] + ("..." if len(doc2) > max_chars else "")
    
    prompt = f"""You are an expert document analyst. Your task is to analyze two documents and identify key data elements present in each.

Document 1:
{doc1_truncated}

Document 2:
{doc2_truncated}

Please analyze both documents and identify:
1. Key data elements (e.g., version numbers, dates, entity names, specifications)
2. Common elements that appear in both documents
3. Elements that are unique to each document
4. Any significant differences in how similar information is presented

Provide your analysis in a structured format."""
    
    return prompt

# prompt use

## zero shot

In [6]:
# Define the nine input combinations
test_combinations = [
    ("cis-r1.pdf", "cis-r1.pdf"),
    ("cis-r1.pdf", "cis-r2.pdf"),
    ("cis-r1.pdf", "cis-r3.pdf"),
    ("cis-r1.pdf", "cis-r4.pdf"),
    ("cis-r2.pdf", "cis-r2.pdf"),
    ("cis-r2.pdf", "cis-r3.pdf"),
    ("cis-r2.pdf", "cis-r4.pdf"),
    ("cis-r3.pdf", "cis-r3.pdf"),
    ("cis-r3.pdf", "cis-r4.pdf"),
]
ZeroShotOutputList = []

pdf_dir = "PDFS"

# Test each combination
for i, (pdf1, pdf2) in enumerate(test_combinations, 1):
    print(f"\nInput-{i}: {pdf1} and {pdf2}")
    pdf1_path = os.path.join(pdf_dir, pdf1)
    pdf2_path = os.path.join(pdf_dir, pdf2)
    doc1, doc2 = input_documents(pdf1_path, pdf2_path)
    print(f"Successfully loaded both documents")
    prompt = zero_shot_prompt(doc1, doc2)
    promptoutput = pipe(prompt)[0]['generated_text'] # make sure to use this! (outputs are actually a dictionary and its silly, but this gets around that)
    print("prompt generated, saving to LLMoutputs.txt")
    ZeroShotOutputList.append(promptoutput) # used for KDE

    PrintPrompt(prompt, "zero Shot", promptoutput)

    torch.cuda.empty_cache() #clean up memory (we may need it!)
    gc.collect()
print(f"\nAll {len(test_combinations)} zero shot outputs saved to zero_shot_outputs.txt")
        


Input-1: cis-r1.pdf and cis-r1.pdf


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Successfully loaded both documents


/home/viper/Documents/GitHub/COMP5700-Project/.venv/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


prompt generated, saving to LLMoutputs.txt
print successfull

Input-2: cis-r1.pdf and cis-r2.pdf


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Successfully loaded both documents
prompt generated, saving to LLMoutputs.txt
print successfull

Input-3: cis-r1.pdf and cis-r3.pdf


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Successfully loaded both documents


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.81 GiB. GPU 0 has a total capacity of 7.65 GiB of which 1.72 GiB is free. Process 161178 has 12.29 MiB memory in use. Including non-PyTorch memory, this process has 4.89 GiB memory in use. Of the allocated memory 3.90 GiB is allocated by PyTorch, and 807.49 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

# key data elements

In [ ]:
def KDE(prompt: str) -> str:
    """
    Construct a zero-shot prompt to identify key data elements in two input documents.
    
    Args:
        doc1: Text content of the first document
        doc2: Text content of the second document
        max_chars: Maximum characters per document to avoid memory issues (default: 2000)
    
    Returns:
        A string containing the zero-shot prompt for the LLM
    """
    # Truncate documents significantly to avoid OOM errors (kept incase needed later)
    # doc1_truncated = doc1[:max_chars] + ("..." if len(doc1) > max_chars else "")
    # doc2_truncated = doc2[:max_chars] + ("..." if len(doc2) > max_chars else "")
    
    prompt = f"""You are an expert document analyst. Your task is to analyze the prompt given identify key data elements present in it.
prompt:
{prompt}

Please analyze the prompt given and generate a output in yaml format like so:
- element1:
  - name:
  - requirements: 
    - req1
    - req2 
    - req3 
"""
    
    return prompt